# SYDE 556/750 &mdash; Practice Notebook 2
## Temporal Representation &amp; Feedforward Transformations

**Prepares you for: Test 2.**

This notebook is **ungraded practice**. Working through it is how you study for Test 2. which is a 20-minute, closed-book, pen-and-paper test that draws directly on the skills you build here.

**How to use it**
- **No solutions are released.** Each part ends with an *Expected result* checkpoint so you can tell whether you are on track.
- Cells marked &#x270D; are for written answers.

**Contents**
1. Generating a random input signal
2. Simulating a spiking neuron
3. Simulating two spiking neurons
4. Computing an optimal filter
5. Using post-synaptic currents as a filter
6. Decoding from a population
7. Connecting groups of neurons (transformations)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(556)   # global RNG; signal generator below takes its own seed


# 1. Generating a random input signal

Write a function `generate_signal(T, dt, rms, limit, seed)` that returns a randomly varying signal $x(t)$ drawn from **band-limited white noise**, together with its Fourier-domain representation $X(\omega)$.

Inputs:
- `T` &mdash; signal length (s)
- `dt` &mdash; time step (s)
- `rms` &mdash; root-mean-square power of the signal
- `limit` &mdash; maximum frequency (Hz)
- `seed` &mdash; RNG seed (so the same signal can be regenerated)

The RMS of a continuous signal of length $T$ is
$$\sigma = \sqrt{\tfrac{1}{T}\int_0^T x(t)^2\, dt}.$$


In [ ]:
def generate_signal(T, dt, rms, limit, seed):
    # Return (x_t, X): the real time-domain signal and its Fourier coefficients.
    # Outline:
    #   - build the frequency axis
    #   - sample the real and imaginary parts of X from a normal distribution
    #   - enforce X(omega) = conj(X(-omega)) so that x(t) comes out purely real
    #   - zero any coefficient whose frequency exceeds `limit`
    #   - inverse-transform to get x(t), then rescale so its RMS equals `rms`
    # TODO
    pass


**a) Time-domain signals.** Plot $x(t)$ for three signals with `limit` = 5, 10, and 20 Hz. For each use `T=1`, `dt=0.001`, `rms=0.5`.

&#x1F40D; See the numpy FFT routines for the forward/inverse transforms.
&#x1F4CC; The transform maps between $t$ and $\omega$, where $\omega$ is in **radians** (not Hz) and $\Delta\omega = 2\pi/T$.


In [ ]:
# Generate and plot x(t) for limit = 5, 10, 20 Hz.
# TODO


*Expected:* larger `limit` produces faster wiggles; all three share a similar overall amplitude (same RMS).

**b) Average power spectrum.** Plot the average $|X(\omega)|$ over 100 signals generated with `T=1`, `dt=0.001`, `rms=0.5`, `limit=10` (each with a different seed). Label the $x$-axis "$\omega$ in radians".


In [ ]:
# Average |X(omega)| over 100 signals; plot vs omega (radians).
# TODO


*Expected:* the average spectrum is roughly flat out to the limit, then drops to ~0 beyond it.

# 2. Simulating a spiking neuron

Simulate a single **Leaky Integrate-and-Fire (LIF)** neuron. The membrane equation is
$$\frac{dv}{dt} = \frac{1}{\tau_{RC}}\,(J - v),$$
which is normalized so the resting voltage is 0 and the threshold is $v_{\mathrm{th}} = 1$. Integrate numerically (Euler). When $v$ reaches threshold the neuron **spikes**: reset $v$ to 0 and hold it there for the refractory period $\tau_{\mathrm{ref}}$. If $v$ ever goes below 0, clamp it to 0. Use $\tau_{RC} = 20$ ms and $\tau_{\mathrm{ref}} = 2$ ms.

The input current is $J = \alpha\,\langle e, x\rangle + J^{\mathrm{bias}}$. Set $e = 1$, and choose $\alpha$ and $J^{\mathrm{bias}}$ so the firing rate is 40 Hz at $x = 0$ and 150 Hz at $x = 1$, using the LIF rate approximation (from PN1 &sect;1.3):
$$G[J] = \frac{1}{\tau_{\mathrm{ref}} - \tau_{RC}\ln\!\left(1 - \tfrac{1}{J}\right)}.$$


In [ ]:
def lif_spikes(x_t, dt, alpha, J_bias, tau_rc=0.020, tau_ref=0.002, e=1.0):
    # Euler-integrate the membrane equation.
    # Spike when v crosses threshold (=1): record the spike time, reset v, and
    # hold through the refractory period. Clamp v at 0 from below.
    # Return the spike train (and, if you like, the voltage trace).
    # TODO
    pass

# Solve for alpha, J_bias from the two-rate condition (40 Hz at x=0, 150 Hz at x=1)
# using the LIF rate equation above or your code from the previous practice notebook.
# TODO


**a) Spike plots for constant inputs.** Plot the spike output for a constant input $x = 0$ over 1 second, and report the number of spikes. Do the same for $x = 1$. Use $\Delta t = 0.001$ s.


In [ ]:
# Constant-input spike rasters for x = 0 and x = 1; report spike counts.
# TODO


*Expected:* roughly 40 spikes at $x=0$ and roughly 150 at $x=1$ over the second (not exact, see part b).

**b) Discussion.** Does the observed number of spikes match the expected number for $x=0$ and $x=1$? Why or why not? What aspects of the simulation affect this accuracy?

&#x270D; *Your answer here.*


**c) Spike plot for a white-noise input.** Plot the spike output for an $x(t)$ from your &sect;1 function with `T=1`, `dt=0.001`, `rms=0.5`, `limit=30`. Overlay $x(t)$.


In [ ]:
# Spikes for a white-noise x(t), with x(t) overlaid.
# TODO


**d) Voltage over time.** Using the same $x(t)$ as in (c), plot the neuron's voltage over the first 0.2 s, with the spikes marked.


In [ ]:
# Voltage trace over the first 0.2 s, with spikes marked.
# TODO


*Expected:* the voltage ramps toward threshold, resets at each spike, and pauses for the refractory period.

# 3. Simulating two spiking neurons

Now simulate **two** neurons with identical parameters except their encoders: one with $e = +1$, the other with $e = -1$. Otherwise use exactly the settings from &sect;2.


**a) Constant inputs.** Plot $x(t)$ and the spike output for $x(t) = 0$ (both neurons should fire at about 40 spikes/s) and, separately, for $x(t) = 1$ (one neuron near 150 spikes/s, the other silent).


In [ ]:
# Two-neuron rasters for constant x = 0 and x = 1.
# TODO


**b) Sinusoidal input.** Plot $x(t)$ and the spike output for $x(t) = \tfrac{1}{2}\sin(10\pi t)$.


In [ ]:
# Two-neuron raster for a sinusoidal input.
# TODO


**c) White-noise input.** Plot $x(t)$ and the spike output for a random signal from &sect;1 with `T=2`, `dt=0.001`, `rms=0.5`, `limit=5`. (Keep this signal, you will decode it in &sect;4 and &sect;5.)


In [ ]:
# Two-neuron raster for a white-noise input (save this signal + spikes for later).
# TODO


# 4. Computing an optimal filter

Decode the spike train from &sect;3c using the **optimal linear filter**. Work with the response $r(t)$ formed from the two neuron spike trains (i.e., the positive-encoder spikes minus the negative-encoder spikes).

In the frequency domain, the optimal filter that minimizes the mean-squared decoding error is
$$H(\omega) = \frac{\langle X(\omega)\,R^{*}(\omega)\rangle}{\langle |R(\omega)|^{2}\rangle},$$
where $\langle\cdot\rangle$ denotes an average (e.g., over windows or trials) and $R^{*}$ is the complex conjugate. The time-domain filter is $h(t) = \mathcal{F}^{-1}\{H(\omega)\}$.


**a) Compute the optimal filter** $h(t)$ for the signal you generated in &sect;3c.


In [ ]:
# Build r(t) from the two spike trains; estimate H(omega) from the cross-spectrum
# and response power spectrum (be sure to apply a gaussian window, sigma = 0.025); inverse-transform to h(t).
# TODO


**b) Plot the optimal filter** in both the time and frequency domains (use sensible $x$-axis limits).

&#x1F4D6; Compare with Figure 4.8 in the book.


In [ ]:
# Time and frequency plots of h(t) / H(omega).
# TODO


**c) Decoded signal.** Plot the original $x(t)$, the spikes, and the decoded $\hat{x}(t)$ (the response filtered by $h$).

&#x1F4D6; Compare with Figure 4.9 in the book.


In [ ]:
# Plot x(t), spikes, and decoded x_hat(t).
# TODO


**d) Power spectra.** Plot the signal $|X(\omega)|$, the response $|R(\omega)|$, and the filtered $|\hat{X}(\omega)|$ power spectra.

&#x1F4D6; Compare with Figure 4.10 in the book.


In [ ]:
# Power spectra of X, R, and X_hat.
# TODO


**e) Discussion.** How do these spectra relate to the optimal filter?

&#x270D; *Your answer here.*


# 5. Using post-synaptic currents as a filter

Instead of the optimal filter, decode with a **post-synaptic current (PSC)** filter
$$h(t) = \begin{cases} c^{-1}\, t^{n}\, e^{-t/\tau} & t \ge 0 \\ 0 & \text{otherwise,}\end{cases}$$
where $n$ is a non-negative integer and $c = \int_0^\infty t^{n} e^{-t/\tau}\, dt$ normalizes the filter to unit area.


**a) Filter vs. $n$.** Plot the normalized $h(t)$ for $n = 0, 1, 2$ with $\tau = 7$ ms.

In [ ]:
# Plot normalized h(t) for n = 0, 1, 2 (tau = 7 ms).
# TODO


**b) Discussion.** What two things do you expect increasing $n$ will do to $\hat{x}(t)$?

&#x270D; *Your answer here.*


**c) Filter vs. $\tau$.** Plot the normalized $h(t)$ for $\tau = 2, 5, 10, 20$ ms with $n = 0$.

In [ ]:
# Plot normalized h(t) for tau = 2, 5, 10, 20 ms (n = 0).
# TODO


**d) Discussion.** What two things do you expect increasing $\tau$ will do to $\hat{x}(t)$?

&#x270D; *Your answer here.*


**e) Decoding with the PSC filter.** Decode $\hat{x}(t)$ from the &sect;3c spikes using $h(t)$ with $n = 0$, $\tau = 7$ ms. Generate the spikes, filter them with $h(t)$, and use the filtered activity to compute your decoder. Plot the filter (time and frequency), and plot $x(t)$, the spikes, and $\hat{x}(t)$.

&#x1F4CC; Do **not** use a built-in convolution. Implement it in the time domain: add a copy of the filter at each spike time and sum (this is the post-synaptic current arriving on each spike).


In [ ]:
# Decode x_hat(t) using a PSC filter (n=0, tau=7 ms); time-domain convolution only.
# TODO


**f) Low-frequency signal.** Using the same decoder and $h(t)$ as in (e), generate a new $x(t)$ with `limit=2` and decode it. Plot $x(t)$, the spikes, and $\hat{x}(t)$.


In [ ]:
# Decode a new low-frequency (limit = 2 Hz) signal with the same filter/decoder.
# TODO


**g) Discussion.** How do the decodings from (e) and (f) compare? Explain.

&#x270D; *Your answer here.*


# 6. Decoding from a population

Build a population of **20 LIF neurons** representing a 1-D value with radius $r = 2$. Use $\tau_{\mathrm{ref}} = 2$ ms, $\tau_{RC} = 20$ ms; draw maximum firing rates uniformly from $[100, 200]$ Hz (with the max rate occurring at $\langle x, e\rangle = r = 2$) and $x$-intercepts uniformly from $[-2, 2]$.

It is easiest to compute decoders in rate mode and reuse them for spiking. With $A$ the rate-mode activity matrix,
$$D^{T} = \big(A A^{T} + N\sigma^{2} I\big)^{-1} A X^{T}.$$
These decoders also work when you simulate spikes (from part c on), but you have to scale the output by the time step `dt`.


**a) Tuning curves.** Plot the tuning curves (firing rate vs. $x \in [-2, 2]$).

In [ ]:
# Build the 20-neuron population; plot tuning curves over x in [-2, 2].
# TODO


**b) Decoder and error.** Compute the decoders accounting for noise ($\sigma = 0.1\cdot 200$ Hz). When computing $\hat{x}$, add Gaussian noise of the same $\sigma$ to the activity. Plot $(x - \hat{x})$ and report the RMSE.


In [ ]:
# Compute noise-aware decoders; decode with noise added; plot (x - x_hat); report RMSE.
# TODO


**c) Decoding a time-varying signal.** Feed a random $x(t)$ (`T=1`, `dt=0.001`, `rms=1`, `limit=5`) into the population, generate spikes, and decode $\hat{x}(t)$ using a PSC filter with $\tau = 5$ ms. Plot $x(t)$ and $\hat{x}(t)$, and report the RMSE.


In [ ]:
# Feed x(t) into the spiking population; decode x_hat(t) with a tau=5 ms PSC filter.
# TODO


*Expected:* $\hat{x}(t)$ tracks $x(t)$ with a small synaptic lag and some spiking jitter.

**(Optional) d) Error vs. neuron count.** Repeat the time-varying decode for $n = 8, 16, 32, 64, 128, 256$ neurons (intercepts in $[-2,2]$, rates $[100,200]$ Hz, encoders $\pm 1$). Plot RMSE vs. $n$ on log-log axes, averaging over at least 5 random populations per $n$.

&#x1F4D6; Compare with Figure 5.3 in the book.


In [ ]:
# (Optional) RMSE vs. neuron count, log-log.
# TODO


# 7. Connecting groups of neurons

Now connect populations so the network **computes a function**. Use populations with radius $r = 1$, intercepts in $[-1, 1]$, and 200 neurons each. Compute the decoders that work over the whole input range. Do **not** recompute decoders for a particular input.

## 7.1 Two groups: $y = 2x + 1$

Group 1 represents $x$; group 2 represents $y$. You need two decoders: one to decode $f(x) = 2x + 1$ from group 1, and the standard identity decoder $f(y) = y$ from group 2. Feed $x(t)$ into group 1, generate spikes, decode $2x+1$, feed that into group 2, and decode $\hat{y}(t)$.

&#x1F4CC; Make sure the maximum firing rates now occur at $\pm 1$ (radius 1).


**a)** Input $x(t) = t - 1$ for 1 s (a ramp from $-1$ to $0$). Plot the ideal $x(t)$ and $y(t)$ along with $\hat{y}(t)$.

In [ ]:
# Ramp input; plot ideal x(t), ideal y(t), and decoded y_hat(t).
# TODO


**b)** Repeat (a) with a step input: ten random values in $[-1, 0]$, each held for 0.1 s.

In [ ]:
# Random step input.
# TODO


**c)** Repeat (a) with $x(t) = 0.2\sin(6\pi t)$.

In [ ]:
# Sinusoidal input.
# TODO


**d) Discussion.** Does the output match the ideal? What kinds of deviations do you see, and why?

&#x270D; *Your answer here.*


## 7.2 Three groups: $z = 2y + 0.5x$

Use three populations. Decode $f(y) = 2y$ from the $y$ group and $f(x) = 0.5x$ from the $x$ group, add the two decoded outputs together, and feed the sum into the $z$ group.


**a)** Sinusoidal inputs $x(t) = \cos(3\pi t)$ and $y(t) = 0.5\sin(2\pi t)$ over 1 s. Plot $x(t)$, $y(t)$, the ideal $z(t)$, and the decoded $\hat{z}(t)$.

In [ ]:
# Three-group network; sinusoidal inputs.
# TODO


**b)** Random inputs: $x(t)$ with `limit=8`, `rms=1`; $y(t)$ with `limit=5`, `rms=0.5`. Plot $x$, $y$, ideal $z$, and $\hat{z}$.

In [ ]:
# Three-group network; random inputs.
# TODO


---
*End of Practice Notebook 2.* Test 2 will assume you can do everything above by hand or from scratch.
